In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Lokální testovací režim Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
Používej lokální testovací režim (dostupný od `qiskit-ibm-runtime` v0.22.0 nebo novějšího) k testování programů před jejich doladěním a odesláním na skutečný kvantový hardware. Po ověření programu v lokálním testovacím režimu stačí pouze změnit název Backend, aby program běžel na QPU.

Chceš-li použít lokální testovací režim, zadej jeden z falešných backendů z ``qiskit_ibm_runtime.fake_provider`` nebo zadej Backend Qiskit Aer při vytváření instance primitivu Qiskit Runtime nebo Session.

- **Fake backends**: [Fake backends](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) v ``qiskit_ibm_runtime.fake_provider`` napodobují chování IBM&reg; QPU pomocí snímků QPU. Snímky QPU obsahují důležité informace o QPU, jako je mapa propojení, základní Gate a vlastnosti Qubitů, které jsou užitečné pro testování Transpileru a provádění simulací QPU s šumem. Model šumu ze snímku je během simulace aplikován automaticky.
- **Simulátor Aer**: Simulátory z [Qiskit Aer](/guides/simulate-with-qiskit-aer) poskytují výkonnější simulaci, která zvládne větší Circuit a [vlastní modely šumu](/guides/build-noise-models). Při použití `AerSimulator` v lokálním testovacím režimu jsou k dispozici různé možnosti metod simulace. Viz [příklad simulačního režimu Clifford](#clifford-sim), který ukazuje, jak efektivně simulovat Cliffordovy Circuit s velkým počtem Qubitů.

    <details>
    <summary>**Seznam simulačních metod dostupných v Qiskit Aer**</summary>

    Více informací najdeš v dokumentaci [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator).

    * ``"automatic"``: Výchozí simulační metoda. Simulační metoda se vybírá automaticky na základě Circuit a modelu šumu.

    * ``"statevector"``: Hustá simulace stavového vektoru, která může vzorkovat výsledky měření z *ideálních* Circuit s veškerými měřeními na konci Circuit. Pro simulace se šumem každý výstřel vzorkuje náhodně vybraný zašuměný Circuit z modelu šumu.

    * ``"density_matrix"``: Simulace matice hustoty, která může vzorkovat výsledky měření ze *zašuměných* Circuit s veškerými měřeními na konci Circuit.

    * ``"stabilizer"``: Efektivní simulátor Cliffordova stabilizátorového stavu, který dokáže simulovat zašuměné Cliffordovy Circuit, pokud jsou všechny chyby v modelu šumu také Cliffordovy chyby.

    * ``"extended_stabilizer"``: Přibližný simulátor pro Clifford + T Circuit založený na rozkladu stavu do stabilizátorového stavu s hodností. Počet členů roste s počtem non-Cliffordových (T) Gate.

    * ``"matrix_product_state"``: Tenzorově-síťový simulátor stavového vektoru, který pro reprezentaci stavu používá reprezentaci maticového součinového stavu (MPS). Lze to provádět s oříznutím nebo bez oříznutí dimenzí vazby MPS v závislosti na nastavení simulátoru. Výchozí chování je bez oříznutí.

    * ``"unitary"``: Hustá simulace unitární matice ideálního Circuit. Tato metoda simuluje unitární matici samotného Circuit, nikoli vývoj počátečního kvantového stavu. Tato metoda dokáže simulovat pouze Gate; nepodporuje měření, reset ani šum.

    * ``"superop"``: Hustá simulace matice superoperátoru ideálního nebo zašuměného Circuit. Tato metoda simuluje matici superoperátoru samotného Circuit, nikoli vývoj počátečního kvantového stavu. Tato metoda dokáže simulovat ideální i zašuměné Gate a resety, ale nepodporuje měření.

    * ``"tensor_network"``: Simulace na bázi tenzorové sítě, která podporuje jak stavový vektor, tak matici hustoty. V současnosti je k dispozici pouze pro GPU a je akcelerována pomocí API cuQuantum `cuTensorNet`.
    </details>

> **Note:** - V lokálním testovacím režimu můžeš zadat všechny možnosti Qiskit Runtime. Všechny možnosti kromě shots jsou však při spuštění na lokálním simulátoru ignorovány.
> - Před použitím fake backends nebo simulátorů Aer se doporučuje nainstalovat Qiskit Aer spuštěním `pip install qiskit-aer`. Fake backends pod kapotou používají simulátory Aer, pokud jsou dostupné, aby využily jejich výkon.


## Příklad s fake backends

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## Příklady s AerSimulator
Příklad se Session, bez šumu:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Chceš-li simulovat se šumem, zadej QPU (kvantový hardware) a odešli ho do Aer. Aer sestaví model šumu na základě kalibračních dat z tohoto QPU a vytvoří instanci Aer Backend s tímto modelem. Pokud preferuješ, můžeš si [sestavit vlastní model šumu](/guides/build-noise-models).

> **Caution:** Na QPU může působit různé druhy šumu. Model šumu Qiskit Aer použitý zde simuluje pouze část z nich, a proto bude pravděpodobně méně závažný než šum na skutečném QPU.
> 
> Podrobnosti o tom, jaké chyby jsou zahrnuty při inicializaci modelu šumu z QPU, najdeš v referenci API Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

Příklad se šumem:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Cliffordova simulace
Protože Cliffordovy Circuit lze efektivně simulovat s ověřitelnými výsledky, je Cliffordova simulace velmi užitečným nástrojem. Podrobný příklad najdeš v [Efektivní simulace stabilizátorových Circuit s primitivy Qiskit Aer](/guides/simulate-stabilizer-circuits).

Příklad:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Další kroky
> **Tip:** - Prostuduj si podrobné [příklady primitivů.](/guides/primitives-examples)
>     - Přečti si [Migrace na primitivy V2](/guides/v2-primitives).
>     - Procvič si primitivy v lekci [Funkce nákladů](/learning/courses/variational-algorithm-design/cost-functions) v IBM Quantum Learning.
>     - Nauč se transpilovat lokálně v sekci [Transpiler](/guides/transpile).
>     - Vyzkoušej tutoriál [Porovnání nastavení Transpileru](/guides/circuit-transpilation-settings#compare-transpiler-settings).